In [2]:
clear_train_dir = False

DATASET = "EDABK_HGR"
# num_label = 20
# EPOCHS = 300

num_label = 6
EPOCHS = 300

BATCH_SIZE = 64

# list of layers: [numcores; axon; neuron]
# model_configs = [[13, 238, 64], [4, 208 ,64], [1, 256, 240]]
model_configs = [[3, 216, 216], [4, 162 ,64], [1, 256, 252]]

# list of config for each layer: [thres, activation_factor]
network_activation = [[0,1], [0,1], [0,1]]

# Path

In [3]:
# Set up base dirs
import os

ROOT_DIR = os.getcwd()

SOFT_DIR=ROOT_DIR+"/Software"
HARD_DIR=ROOT_DIR+"/Hardware"

DATASET_DIR = SOFT_DIR+"/data/processed/"
TRAIN_DIR=SOFT_DIR+"/training"
LOG_DIR=SOFT_DIR+"/log/"+DATASET

os.makedirs(LOG_DIR, exist_ok=True)

# Install

In [4]:
!pip install tensorflow
!pip install keras

In [5]:
%cd {SOFT_DIR}

if (clear_train_dir):
    !rm -rf {TRAIN_DIR}
    !unzip "training.zip"

/home/nam/NPLink/SNN_framework/Software


/home/nam/anaconda3/envs/duongk65/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [6]:
%cd {TRAIN_DIR}
!pip install "./tealayers/tealayer4.0"
!pip install "./edabkutils"

/home/nam/NPLink/SNN_framework/Software/training


Processing ./tealayers/tealayer4.0
  Preparing metadata (setup.py) ... done
  DEPRECATION: Building 'tealayer4' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'tealayer4'. Discussion can be found at https://github.com/pypa/pip/issues/6334
  Created wheel for tealayer4: filename=tealayer4-4.0-py3-none-any.whl size=7049 sha256=9aa78b3f165c06fb4576d3bcb422a69ed9530fa9ccb3e4e5c90eced7dfc7cd74
  Stored in directory: /tmp/pip-ephem-wheel-cache-qm5whnhb/wheels/1a/26/d3/5dd2f654ae20b74243fd57ec055525fcb8f9bcf2c4765c160e
Successfully built tealayer4
  Attempting uninstall: tealayer4
    Found existing installation: tealayer4 4.0
    Uninstalling tealayer4-4.0:
      Successfully uninstalled tealayer4-

# Train

## Prepare and import package

In [7]:
import tensorflow as tf

from tealayer4 import Tea, AdditivePooling, tea_weight_initializer
from tensorflow.keras.layers import Flatten, Activation, Input, Lambda, Concatenate
from tensorflow.keras.losses import CategoricalFocalCrossentropy,BinaryCrossentropy, CategoricalCrossentropy
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical, plot_model
from tensorflow.keras import Model
from tensorflow.keras.callbacks import EarlyStopping, LearningRateScheduler, ModelCheckpoint

import json
import yaml
import numpy as np
import math
from tensorflow.keras.optimizers import Adam
from sklearn.utils import class_weight

from edabkutils.modelize import auto_train_config, save_configure_json, get_configs, get_core_arrange, write_config_sim


# from tealayer3 import Tea, AdditivePooling
# from tensorflow.keras.losses import CategoricalFocalCrossentropy,BinaryCrossentropy, CategoricalCrossentropy
# from tensorflow.keras.optimizers import Adam
# from tensorflow.keras.utils import to_categorical, plot_model
# from tensorflow.keras import Model
# from tensorflow.keras.callbacks import EarlyStopping, LearningRateScheduler, ModelCheckpoint
# from tensorflow.keras.layers import Flatten, Activation, Input, Lambda, Concatenate
# from tensorflow.keras.datasets import mnist
# from tensorflow.keras.optimizers import Adam
# from tensorflow.keras.utils import to_categorical, plot_model
# from tensorflow.keras import Model
# from tensorflow.keras.callbacks import EarlyStopping
# import numpy as np
# import math
# import cv2
# import tensorflow.compat.v1 as tf
# tf.disable_v2_behavior()
# from tensorflow.keras.optimizers.legacy import Adam
# from edabkutils.modelize import auto_train_config, save_configure_json, get_configs, get_core_arrange, write_config_sim

# from sklearn.utils import class_weight

2026-03-10 17:46:20.288306: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-10 17:46:21.263641: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [8]:
number_of_layers = len(model_configs)
[x_range, y_range, core_arrange] = get_core_arrange(model_configs)

print("Model configurations:", model_configs)
print("Core arrangement:", core_arrange)

Model configurations: [[3, 216, 216], [4, 162, 64], [1, 256, 252]]
Core arrangement: [[[0, 0], [1, 0], [2, 0]], [[0, 1], [1, 1], [2, 1], [3, 1]], [[0, 2]]]


In [9]:
data = np.load(DATASET_DIR+DATASET+".npz")

X_train = data["X_train"]
y_train = data["y_train"]

X_test = data["X_test"]
y_test = data["y_test"]

class_weights = class_weight.compute_class_weight(class_weight='balanced',
                                                 classes=np.unique(y_train),
                                                 y=y_train)
print(f"Class weight: {class_weights}")

y_train = to_categorical(y_train, num_label)
y_test = to_categorical(y_test, num_label)

X_train = np.expand_dims(X_train, axis=-1)
X_test = np.expand_dims(X_test, axis=-1)

Class weight: [1.01499423 1.02325581 1.03651355 0.96069869 0.98765432 0.98104794]


## Train model

In [ ]:
# Initial the SNN network

# Shape the input to right size
inputs = Input(shape=(X_train.shape[1:]))
# print(f"inputs = Input(shape={(X_train.shape[1:])})")
# print(f"core_size = {model_configs[0][1]}")

# Flatten the inputs
flattened_inputs = Flatten()(inputs)

layer_input = flattened_inputs
# For loop for each layer
for layer_ind in range(number_of_layers-1):
  layer = []
  # For each core in each layer
  for core_ind in range(model_configs[layer_ind][0]):
    core = Lambda(lambda x, start=model_configs[layer_ind][1]*core_ind, end=model_configs[layer_ind][1]*(core_ind+1): x[:, start:end])(layer_input)
    core = Tea(units=model_configs[layer_ind][2], threshold = network_activation[layer_ind][0], activation_factor = network_activation[layer_ind][1], name=f'tea_{layer_ind}_{core_ind}')(core)
    # core = Tea(units=model_configs[layer_ind][2], name=f'tea_{layer_ind}_{core_ind}')(core)
    # print(f"core = Tea(units={model_configs[layer_ind][2]}, threshold = {network_activation[layer_ind][0]}, activation_factor = {network_activation[layer_ind][1]}, name=f'tea_{layer_ind}_{core_ind}')(core)")
    layer.append(core)

  layer_input = Concatenate(axis=1)(layer)
  # print(f"layer_input = Concatenate(axis=1)({layer})")

# core = Tea(units=model_configs[number_of_layers-1][2], name=f'tea_{number_of_layers-1}')(layer_input)
core = Tea(units=model_configs[number_of_layers-1][2], threshold=network_activation[number_of_layers-1][0], activation_factor=network_activation[number_of_layers-1][1], name=f'tea_{number_of_layers-1}')(layer_input)
# print(f"core = Tea(units={model_configs[number_of_layers-1][2]}, threshold={network_activation[number_of_layers-1][0]}, activation_factor={network_activation[number_of_layers-1][1]}, name=f'tea_{number_of_layers-1}')(layer_input)")
network = AdditivePooling(num_label)(core)

2026-03-10 17:46:23.206865: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-03-10 17:46:23.280823: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-03-10 17:46:23.281073: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-

AttributeError: Exception encountered when calling Tea.call().

[1mCould not automatically infer the output shape / dtype of 'tea_0_0' (of type Tea). Either the `Tea.call()` method is incorrect, or you need to implement the `Tea.compute_output_spec() / compute_output_shape()` method. Error encountered:

module 'keras._tf_keras.keras.backend' has no attribute 'learning_phase'[0m

Arguments received by Tea.call():
  • args=('<KerasTensor shape=(None, 216), dtype=float32, sparse=False, ragged=False, name=keras_tensor_4>',)
  • kwargs=<class 'inspect._empty'>

In [ ]:
existing_runs = [
    d for d in os.listdir(LOG_DIR)
    if d.startswith("run_")
]

run_numbers = [int(d.split("_")[1]) for d in existing_runs] if existing_runs else [0]
next_run = max(run_numbers) + 1

run_dir = os.path.join(LOG_DIR, f"run_{next_run:02d}")
checkpoint_dir = os.path.join(run_dir, "checkpoints")

os.makedirs(checkpoint_dir)

print("Run directory:", run_dir)

# Train
predictions = Activation('softmax')(network)

model = Model(inputs=inputs, outputs=predictions)

model.compile(loss=CategoricalCrossentropy(),
              optimizer=Adam(),
              metrics=['accuracy'],
              run_eagerly=True)

def lr_schedule(epoch):
    if epoch <= 80:
        return 0.001
    elif epoch <= 150:
        return 0.0001
    else:
        return 0.00001
reduce_lr = LearningRateScheduler(lr_schedule)

# Using callback EarlyStopping
early_stopping = EarlyStopping(
    monitor='val_accuracy',  # monitor the accuracy of validation set
    patience=200,  # Allow max 5 epoch without improvement
    min_delta=0,
    mode='max',
    verbose=1,
    restore_best_weights=True,
)

checkpoint = ModelCheckpoint(
    filepath=os.path.join(checkpoint_dir, "best_model.keras"),
    monitor="val_accuracy",
    mode="max",
    save_best_only=True,
    verbose=1
)

history = model.fit(
    X_train, y_train,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    verbose=1,
    validation_split=0.2,
    callbacks=[reduce_lr, early_stopping, checkpoint])

print(history.history.keys())
score = model.evaluate(X_test, y_test, verbose=0)

print("Validation Accuracy: ",max(history.history['val_accuracy']))
print("Test Loss: ", score[0])
print("Test Accuracy: ", score[1])

print("\n==========================")
model_path = os.path.join(run_dir, "tea_model.keras")
model.save(model_path)

print("Model saved:", model_path)

metrics = {
    "val_accuracy": float(max(history.history['val_accuracy'])),
    "test_accuracy": float(score[1]),
    "test_loss": float(score[0]),
    "epochs_trained": len(history.history["loss"]),
    "batch_size": BATCH_SIZE,
    "total_epochs": EPOCHS
}

with open(os.path.join(run_dir, "metrics.json"), "w") as f:
    json.dump(metrics, f, indent=4)

Run directory: /home/nam/NPLink/SNN_framework/Software/log/EDABK_HGR/run_11
Epoch 1/300


66/66 ━━━━━━━━━━━━━━━━━━━━ 0s 149ms/step - accuracy: 0.1710 - loss: nan
Epoch 1: val_accuracy improved from None to 0.15341, saving model to /home/nam/NPLink/SNN_framework/Software/log/EDABK_HGR/run_11/checkpoints/best_model.keras
66/66 ━━━━━━━━━━━━━━━━━━━━ 14s 165ms/step - accuracy: 0.1693 - loss: nan - val_accuracy: 0.1534 - val_loss: 1.7918 - learning_rate: 0.0010
Epoch 2/300
66/66 ━━━━━━━━━━━━━━━━━━━━ 0s 152ms/step - accuracy: 0.1725 - loss: nan
Epoch 2: val_accuracy did not improve from 0.15341
66/66 ━━━━━━━━━━━━━━━━━━━━ 11s 167ms/step - accuracy: 0.1669 - loss: nan - val_accuracy: 0.1534 - val_loss: 1.7918 - learning_rate: 0.0010
Epoch 3/300
66/66 ━━━━━━━━━━━━━━━━━━━━ 0s 150ms/step - accuracy: 0.1623 - loss: nan
Epoch 3: val_accuracy did not improve from 0.15341
66/66 ━━━━━━━━━━━━━━━━━━━━ 11s 162ms/step - accuracy: 0.1669 - loss: nan - val_accuracy: 0.1534 - val_loss: 1.7918 - learning_rate: 0.0010
Epoch 4/300
66/66 ━━━━━━━━━━━━━━━━━━━━ 0s 155ms/step - accuracy: 0.1687 - loss: na

KeyboardInterrupt: 

In [ ]:
model.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 9, 72, 1)  │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_2 (Flatten) │ (None, 648)       │          0 │ input_layer_2[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_14 (Lambda)  │ (None, 216)       │          0 │ flatten_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_15 (Lambda)  │ (None, 216)       │          0 │ flatten_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_16 (Lambda)  │ (None, 216)       │          0 │ flatten_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tea_0_0 (Tea)       │ (None, 216)       │     93,528 │ lambda_14[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tea_0_1 (Tea)       │ (None, 216)       │     93,528 │ lambda_15[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tea_0_2 (Tea)       │ (None, 216)       │     93,528 │ lambda_16[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_4       │ (None, 648)       │          0 │ tea_0_0[0][0],    │
│ (Concatenate)       │                   │            │ tea_0_1[0][0],    │
│                     │                   │            │ tea_0_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_17 (Lambda)  │ (None, 162)       │          0 │ concatenate_4[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_18 (Lambda)  │ (None, 162)       │          0 │ concatenate_4[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_19 (Lambda)  │ (None, 162)       │          0 │ concatenate_4[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_20 (Lambda)  │ (None, 162)       │          0 │ concatenate_4[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tea_1_0 (Tea)       │ (None, 64)        │     20,800 │ lambda_17[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tea_1_1 (Tea)       │ (None, 64)        │     20,800 │ lambda_18[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tea_1_2 (Tea)       │ (None, 64)        │     20,800 │ lambda_19[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tea_1_3 (Tea)       │ (None, 64)        │     20,800 │ lambda_20[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_5       │ (None, 256)       │          0 │ tea_1_0[0][0],    │
│ (Concatenate)       │                   │            │ tea_1_1[0][0],    │
│                     │                   │            │ tea_1_2[0][0],    │
│                     │                   │            │ tea_1_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tea_2 (Tea)         │ (None, 252)       │    129,276 │ concatenate_5[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ additive_pooling_2  │ (None, 6)         │          0 │ tea_2[0][0]       │
│ (AdditivePooling)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_2        │ (None, 6)         │          0 │ additive_pooling… │
│ (Activation)        │                   │            │                 

 Total params: 987,278 (3.77 MB)

 Trainable params: 247,108 (965.27 KB)

 Non-trainable params: 245,952 (960.75 KB)

 Optimizer params: 494,218 (1.89 MB)

In [ ]:
plot_model(model, to_file=TRAIN_DIR+'/model_architecture.png', show_shapes=True, show_layer_names=True)


You must install pydot (`pip install pydot`) for `plot_model` to work.
